# 🛡️ Bank Shield AI - Veri Birleştirme ve Keşifçi Analiz (EDA)

Bu notebook, ham veri setlerini (`users`, `cards`, `transactions`) birleştirerek modellemeye hazır tek bir ana tablo (`df_combined`) oluşturur.

**Yapılan İşlemler:**
* Veri tiplerinin temizlenmesi (Amount, Date, Zip).
* Tabloların ID üzerinden ilişkiselleştirilmesi.
* MCC kodları ve Fraud etiketlerinin (Target) eklenmesi.
* Temel zaman özelliklerinin (Hour, Day, Weekend) türetilmesi.

> **⚠️ Teknik Not:** > `transactions_data.csv` dosyası çok büyük olduğu için işlem süreleri uzun sürebilir. 
> Bu notebook'u bir kez çalıştırdığınızda yerelinizde `data/processed/df_combined.parquet` oluşacaktır. 
> Modelleme ve Streamlit aşamalarında bu Parquet dosyasını okuyarak devam edebilirsiniz. 
> (Not: Parquet dosyası büyük olduğu için GitHub'a pushlanmamıştır.)

In [2]:
import pandas as pd
import json

# =========================================================
# 1. DOSYA YOLLARI
# =========================================================
PATH_RAW = '../data/raw/'

# =========================================================
# 2. VERİLERİ OKUMA
# =========================================================
users = pd.read_csv(f'{PATH_RAW}users_data.csv')
cards = pd.read_csv(f'{PATH_RAW}cards_data.csv')

In [3]:
# Büyük dosya olduğu için şimdilik ilk 100.000 satır
transactions = pd.read_csv(f'{PATH_RAW}transactions_data.csv')

with open(f'{PATH_RAW}mcc_codes.json', 'r') as f:
    mcc_codes = json.load(f)

with open(f'{PATH_RAW}train_fraud_labels.json', 'r') as f:
    fraud_labels = json.load(f)

print("✅ Tüm veriler başarıyla yüklendi!")

✅ Tüm veriler başarıyla yüklendi!


In [4]:
# 3. İLK KONTROLLER
# =========================================================
print("\n--- Veri Boyutları ---")
print(f"Users shape: {users.shape}")
print(f"Cards shape: {cards.shape}")
print(f"Transactions shape: {transactions.shape}")

print("\n--- Transactions sütunları ---")
print(transactions.columns.tolist())

print("\n--- Users sütunları ---")
print(users.columns.tolist())

print("\n--- Cards sütunları ---")
print(cards.columns.tolist())



--- Veri Boyutları ---
Users shape: (2000, 14)
Cards shape: (6146, 13)
Transactions shape: (13305915, 12)

--- Transactions sütunları ---
['id', 'date', 'client_id', 'card_id', 'amount', 'use_chip', 'merchant_id', 'merchant_city', 'merchant_state', 'zip', 'mcc', 'errors']

--- Users sütunları ---
['id', 'current_age', 'retirement_age', 'birth_year', 'birth_month', 'gender', 'address', 'latitude', 'longitude', 'per_capita_income', 'yearly_income', 'total_debt', 'credit_score', 'num_credit_cards']

--- Cards sütunları ---
['id', 'client_id', 'card_brand', 'card_type', 'card_number', 'expires', 'cvv', 'has_chip', 'num_cards_issued', 'credit_limit', 'acct_open_date', 'year_pin_last_changed', 'card_on_dark_web']


In [5]:
# 4. TRANSACTIONS TEMİZLİĞİ
# =========================================================

# Amount: '$' işaretini kaldır, float yap
transactions['amount'] = (
    transactions['amount']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)

# Date: datetime yap
transactions['date'] = pd.to_datetime(transactions['date'], errors='coerce')

# Zip: string yap, .0 temizle
transactions['zip'] = (
    transactions['zip']
    .astype(str)
    .str.replace('.0', '', regex=False)
)

# Negatif tutarlar iade olabilir
transactions['is_return'] = transactions['amount'] < 0
transactions['amount'] = transactions['amount'].abs()

print("\n✅ Transactions temizliği tamamlandı!")



✅ Transactions temizliği tamamlandı!


In [6]:
# 5. ID TİPLERİNİ EŞİTLE
# =========================================================
users['id'] = users['id'].astype(str)
cards['id'] = cards['id'].astype(str)
cards['client_id'] = cards['client_id'].astype(str)

transactions['id'] = transactions['id'].astype(str)
transactions['client_id'] = transactions['client_id'].astype(str)
transactions['card_id'] = transactions['card_id'].astype(str)

print("\n--- ID tipleri ---")
print("users['id']:", users['id'].dtype)
print("cards['id']:", cards['id'].dtype)
print("cards['client_id']:", cards['client_id'].dtype)
print("transactions['id']:", transactions['id'].dtype)
print("transactions['client_id']:", transactions['client_id'].dtype)
print("transactions['card_id']:", transactions['card_id'].dtype)


--- ID tipleri ---
users['id']: str
cards['id']: str
cards['client_id']: str
transactions['id']: str
transactions['client_id']: str
transactions['card_id']: str


In [7]:
# 6. TRANSACTIONS + USERS MERGE
# =========================================================
df_combined = pd.merge(
    transactions,
    users,
    left_on='client_id',
    right_on='id',
    how='left',
    suffixes=('', '_user')
)

print("\n✅ Transactions + Users birleştirildi")
print("Shape:", df_combined.shape)


✅ Transactions + Users birleştirildi
Shape: (13305915, 27)


In [8]:
# users tablosundan gelen id sütununu kaldır
if 'id_user' in df_combined.columns:
    df_combined = df_combined.drop(columns=['id_user'])

# Bazı durumlarda merge sonrası users.id direkt 'id_y' olabilir
if 'id_y' in df_combined.columns:
    df_combined = df_combined.drop(columns=['id_y'])


In [9]:
# 7. CARDS TABLOSUNU EKLE
# transactions.card_id  <-> cards.id
# =========================================================
df_combined = pd.merge(
    df_combined,
    cards,
    left_on='card_id',
    right_on='id',
    how='left',
    suffixes=('', '_card')
)

print("\n✅ Cards tablosu eklendi")
print("Shape:", df_combined.shape)

# cards'tan gelen gereksiz tekrar id sütununu kaldır
if 'id_card' in df_combined.columns:
    df_combined = df_combined.drop(columns=['id_card'])

# Eğer merge sonrası 'id_y' oluştuysa temizle
if 'id_y' in df_combined.columns:
    df_combined = df_combined.drop(columns=['id_y'])



✅ Cards tablosu eklendi
Shape: (13305915, 39)


In [10]:
# 8. MCC CODES EKLEME
# =========================================================
# mcc_codes JSON ise dict olarak gelir:
# örnek: {"1711": "Heating, Plumbing...", ...}

mcc_df = pd.DataFrame(list(mcc_codes.items()), columns=['mcc', 'mcc_description'])
mcc_df['mcc'] = mcc_df['mcc'].astype(str)

# transactions içindeki mcc de string olsun
df_combined['mcc'] = df_combined['mcc'].astype(str)

df_combined = pd.merge(
    df_combined,
    mcc_df,
    on='mcc',
    how='left'
)

print("\n✅ MCC açıklamaları eklendi")
print("Shape:", df_combined.shape)



✅ MCC açıklamaları eklendi
Shape: (13305915, 39)


In [11]:
# 9. FRAUD LABELS DÜZELTME VE EKLEME
# =========================================================
# JSON yapısı:
# {
#   "target": {
#       "7476734": "No",
#       "7481767": "Yes",
#       ...
#   }
# }

fraud_labels = fraud_labels['target']

fraud_df = pd.DataFrame(list(fraud_labels.items()), columns=['id', 'is_fraud'])
fraud_df['id'] = fraud_df['id'].astype(str)

# Yes/No -> 1/0
fraud_df['is_fraud'] = fraud_df['is_fraud'].map({'Yes': 1, 'No': 0})

# Fraud label transaction id üzerinden eklenmeli
# Çünkü fraud JSON key = transactions.id
df_combined = pd.merge(
    df_combined,
    fraud_df,
    on='id',
    how='left'
)

# Eksik olanları 0 kabul et
df_combined['is_fraud'] = df_combined['is_fraud'].fillna(0).astype(int)

print("\n✅ Fraud label eklendi")
print("Shape:", df_combined.shape)


✅ Fraud label eklendi
Shape: (13305915, 40)


In [12]:
# 10. TARİH FEATURE'LARI
# =========================================================
df_combined['year'] = df_combined['date'].dt.year
df_combined['month'] = df_combined['date'].dt.month
df_combined['day'] = df_combined['date'].dt.day
df_combined['hour'] = df_combined['date'].dt.hour
df_combined['day_of_week'] = df_combined['date'].dt.day_name()
df_combined['is_weekend'] = df_combined['day_of_week'].isin(['Saturday', 'Sunday']).astype(int)

print("\n✅ Tarih feature'ları oluşturuldu")


✅ Tarih feature'ları oluşturuldu


In [13]:
# 11. KONTROLLER
# =========================================================
print("\n--- Son tablo bilgisi ---")
print(df_combined.shape)

print("\n--- İlk 5 satır ---")
print(df_combined.head())

print("\n--- Fraud dağılımı ---")
print(df_combined['is_fraud'].value_counts(dropna=False))

print("\n--- Fraud oranı ---")
print(df_combined['is_fraud'].mean())

print("\n--- Eksik değer kontrolü (ilk 20 sütun) ---")
print(df_combined.isnull().sum().sort_values(ascending=False).head(20))



--- Son tablo bilgisi ---
(13305915, 46)

--- İlk 5 satır ---
        id                date client_id card_id  amount           use_chip  \
0  7475327 2010-01-01 00:01:00      1556    2972   77.00  Swipe Transaction   
1  7475328 2010-01-01 00:02:00       561    4575   14.57  Swipe Transaction   
2  7475329 2010-01-01 00:02:00      1129     102   80.00  Swipe Transaction   
3  7475331 2010-01-01 00:05:00       430    2860  200.00  Swipe Transaction   
4  7475332 2010-01-01 00:06:00       848    3915   46.41  Swipe Transaction   

   merchant_id merchant_city merchant_state    zip  ... year_pin_last_changed  \
0        59935        Beulah             ND  58523  ...                  2008   
1        67570    Bettendorf             IA  52722  ...                  2015   
2        27092         Vista             CA  92084  ...                  2008   
3        27092   Crown Point             IN  46307  ...                  2006   
4        13051       Harwood             MD  20776  ...  

### 🔍 Eksik Değer Notları:
- **errors:** İşlemlerin %98'inde hata yok, bu yüzden bu alanın boş olması normaldir (NaN = Hata Yok).
- **merchant_state:** Online işlemlerde eyalet bilgisi bulunmadığı için boş kalmıştır.
- **Diğer Sütunlar:** Veri seti genel olarak çok temizdir, kritik alanlarda eksik veri bulunmamaktadır.

In [14]:
df_combined['is_fraud'].unique()

array([0, 1])

In [15]:
df_combined['is_fraud'].sum()

np.int64(13332)

In [16]:
df_combined

,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,...,year_pin_last_changed,card_on_dark_web,mcc_description,is_fraud,year,month,day,hour,day_of_week,is_weekend
0,7475327,2010-01-01 00:01:00,1556,2972,77.00,Swipe Transaction,59935,Beulah,ND,58523,...,2008,No,Miscellaneous Food Stores,0,2010,1,1,0,Friday,0
1,7475328,2010-01-01 00:02:00,561,4575,14.57,Swipe Transaction,67570,Bettendorf,IA,52722,...,2015,No,Department Stores,0,2010,1,1,0,Friday,0
2,7475329,2010-01-01 00:02:00,1129,102,80.00,Swipe Transaction,27092,Vista,CA,92084,...,2008,No,Money Transfer,0,2010,1,1,0,Friday,0
3,7475331,2010-01-01 00:05:00,430,2860,200.00,Swipe Transaction,27092,Crown Point,IN,46307,...,2006,No,Money Transfer,0,2010,1,1,0,Friday,0
4,7475332,2010-01-01 00:06:00,848,3915,46.41,Swipe Transaction,13051,Harwood,MD,20776,...,2014,No,Drinking Places (Alcoholic Beverages),0,2010,1,1,0,Friday,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13305910,23761868,2019-10-31 23:56:00,1718,2379,1.11,Chip Transaction,86438,West Covina,CA,91792,...,2019,No,Miscellaneous Food Stores,0,2019,10,31,23,Thursday,0
13305911,23761869,2019-10-31 23:56:00,1766,2066,12.80,Online Transaction,39261,ONLINE,NaN,NaN,...,2012,No,"Digital Goods - Media, Books, Apps",0,2019,10,31,23,Thursday,0
13305912,23761870,2019-10-31 23:57:00,199,1031,40.44,Swipe Transaction,2925,Allen,TX,75002,...,2007,No,"Utilities - Electric, Gas, Water, Sanitary",0,2019,10,31,23,Thursday,0
13305913,23761873,2019-10-31 23:58:00,1986,5443,4.00,Chip Transaction,46284,Daly City,CA,94014,...,2010,No,"Grocery Stores, Supermarkets",0,2019,10,31,23,Thursday,0


# Bellek tasarrufu için tip dönüşümü (Opsiyonel ama önerilir)
df_combined['amount'] = df_combined['amount'].astype('float32')
df_combined['is_fraud'] = df_combined['is_fraud'].astype('int8')
df_combined['hour'] = df_combined['hour'].astype('int8')

# Veriyi parquet dosyası şeklinde kaydet

> **⚠️ Teknik Not:** > `transactions_data.csv` dosyası çok büyük olduğu için işlem süreleri uzun sürebilir. 
> Bu notebook'u bir kez çalıştırdığınızda yerelinizde `data/processed/df_combined.parquet` oluşacaktır. 
> Modelleme ve Streamlit aşamalarında bu Parquet dosyasını okuyarak devam edebilirsiniz. 
> (Not: Parquet dosyası büyük olduğu için GitHub'a pushlanmamıştır.)

In [ ]:
import os

# 12. VERİYİ KAYDETME (Güvenli Yol)
# =========================================================
output_path = '../data/processed'
os.makedirs(output_path, exist_ok=True)

# İstediğin isimle kaydet
df_combined.to_parquet(f'{output_path}/df_combined.parquet')

print("✅ Dev veri seti 'data/processed/df_combined.parquet' olarak kaydedildi!")

## 🏁 Sonuç
Veri setleri başarıyla birleştirildi ve `data/processed/df_combined.parquet` olarak kaydedildi. 
Bu dosya; 13.3 milyon işlem, 2000 müşteri ve 6146 kart bilgisini içeren, 
modelleme aşamasına (Behavioral Credit Risk & Fraud Detection) hazır ana veri setidir.